# Cross-dataset 02 · D2 -> D3 (zero-shot)

**Rules of this experiment** (docs/cross_dataset/design.md): outputs only in `<outputs>/CROSS_D3_D2_19CH_v1/`;
the release (`FINAL_v1/`), amendment A2 (running Exp 8a), the D3 stores, their lock, pins and decisions are only
read. Source-only training; the target is BLINDED (labels masked) when it is scored; target labels are read only by
the evaluation in notebook 03, after the campaign is frozen. Nothing is selected with target results.

Each unit fits a FRESH model of an unchanged Exp 0 / 1 / 3-7 / 8a variant (fixed 60-epoch protocol for the GNNs;
classical models: inner CV for C inside the source only) on all 23 D2 participants; plus the source-only D2 cross-validation (5 x 5 folds), then scores the BLINDED D3 windows; one mean
P(AD) per participant is saved, without labels. Completed units are checkpointed and skipped on restart. No target
metric is computed here (notebook 03).

**How to use:** *Runtime → Run all*. Allow Google Drive access when asked. If Colab disconnects, open the notebook
again and *Run all* again: finished work is kept on Google Drive and skipped. Everything is written to
`<outputs>/FINAL_v1/` on Drive (never to the temporary Colab disk); the validated graph stores and all earlier
results are only read.
 Use a **GPU runtime** (T4).

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only to move THIS EXPERIMENT's code pin deliberately
                      # (<outputs>/CROSS_D3_D2_19CH_v1/CODE_COMMIT.txt; the release and A2 pins are never used or moved here).
BRANCH = 'claude/cross-dataset-d3-d2-19ch'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py', '--namespace', 'cross'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh as _sh

def sh(cmd, cwd=None):  # every script runs with the file backend: Colab's inline backend breaks figures in scripts
    os.environ['MPLBACKEND'] = 'Agg'
    return _sh(cmd, cwd)

from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
FINAL = OUT + '/FINAL_v1'
print('release outputs on Drive:', FINAL)
CROSS = OUT + '/CROSS_D3_D2_19CH_v1'
print('cross-dataset outputs on Drive:', CROSS)

## Numerical environment (the Python / library versions of the completed Exp 7 units)

The cross-dataset fits must be comparable with Exp 0-8a, so every script runs with the interpreter that reproduces the
versions recorded by a completed Exp 7 unit (python, numpy, scipy, pandas, scikit-learn, torch); if this kernel
differs, `/content/restore<python digits>` is built with `uv` and verified (temporary Colab disk only). Leave
`ACCEPT_ENVIRONMENT_DIFFERENCE = False`.

In [ ]:
ACCEPT_ENVIRONMENT_DIFFERENCE = False
from adgraph import numerical_env
REFERENCE = numerical_env.reference_libraries(OUT, 'exp7_earm_gnn')
PY = numerical_env.ensure_interpreter(REFERENCE, repo='/content/repo', base='/content',
                                      accept_difference=ACCEPT_ENVIRONMENT_DIFFERENCE, run=sh)
DIFF = numerical_env.differences(REFERENCE, numerical_env.interpreter_versions(PY, REFERENCE))
assert ACCEPT_ENVIRONMENT_DIFFERENCE or not DIFF, f'numerical environment differs from Exp 7: {DIFF}'
def cross(args):   # every step of this experiment: scripts/cross_dataset.py with the verified interpreter
    sh(f"{PY} scripts/cross_dataset.py {args}", cwd='/content/repo')

## Training authorization (author) and status

In [ ]:
AUTHORIZED_BY = ''          # the author's name after the written authorization of real-data training
if AUTHORIZED_BY:
    cross(f'authorize --stage train --by "{AUTHORIZED_BY}"')
cross('status')

## Run (resumable; this direction only). `CONTEXTS` is the only cell meant to be edited.

In [ ]:
CONTEXTS = [('lcmv', 'B'), ('wmne', 'B'), ('sensor', 'B'), ('lcmv', 'A'), ('wmne', 'A'), ('sensor', 'A')]
ERRORS = {}
for inverse, mode in CONTEXTS:
    print(f'==== D2_to_D3 | {inverse} | mode {mode}')
    try:
        cross(f'run --direction D2_to_D3 --inverses {inverse} --modes {mode}')
    except RuntimeError as exc:
        ERRORS[(inverse, mode)] = exc
cross('status')
if ERRORS:
    raise RuntimeError(f'units failed in {sorted(ERRORS)}; completed units are saved. Run all again.')